# 02 - Retrospective replay on real public screens

This is the evidence. Everything else in Loopwise is machinery; this notebook
asks whether the machinery pays for itself on data somebody actually measured.

**The method.** Take a public dataset where every compound has been measured.
Hide all the labels. A policy selects a batch; we reveal only that batch's
labels; the model retrains; repeat. Because the full answer key exists, we can
compute what no live campaign can: what each policy *would have* found, on the
same data, with the same model and budget.

**Two datasets, chosen because they disagree:**

| Dataset | What it is | Hit rate | Why it is here |
| --- | --- | --- | --- |
| **Tox21 NR-AR** | 7,258 compounds screened against the androgen receptor | 4.2% | A realistically scarce primary screen |
| **BACE-1** | 1,513 beta-secretase inhibitors with measured IC50 | 44.6% at pIC50 >= 7 | A dose-response set, where potency is continuous |

**Spoiler, stated up front:** on one of these datasets greedy wins on hits, and
on the other greedy wins on hits *and* produces the worst model of any policy.
Neither result is the tidy one. Both are reported.

In [ ]:
import sys
from pathlib import Path

# Figures and tables land in results/ so the committed notebooks and the README
# reference the same files.
ROOT = Path.cwd()
if not (ROOT / "loopwise").exists():
    ROOT = ROOT.parent
FIGDIR = ROOT / "results" / "figures"
TABDIR = ROOT / "results" / "tables"
FIGDIR.mkdir(parents=True, exist_ok=True)
TABDIR.mkdir(parents=True, exist_ok=True)
print("loopwise", __import__("loopwise").__version__, "| figures ->", FIGDIR)

In [ ]:
import warnings

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.ensemble import RandomForestClassifier

from loopwise.datasets import load_bace, load_tox21
from loopwise.replay import compare_policies, plot_comparison

warnings.filterwarnings("ignore", category=RuntimeWarning)
plt.rcParams.update({"figure.dpi": 110, "font.size": 9})

POLICIES = ["random", "greedy", "uncertainty", "bald", "thompson", "hybrid", "kdpp"]
SEEDS = (0, 1, 2, 3, 4)
def forest():
    return RandomForestClassifier(n_estimators=100, random_state=0, n_jobs=2)

## Dataset 1: Tox21 NR-AR

The Tox21 Data Challenge (2014) screened ~7,800 compounds against 12
nuclear-receptor and stress-response assays. We use the androgen receptor
(NR-AR) assay as a binary screen: 4.2% of the 7,258 measured compounds are
active. That scarcity is the point -- it is the regime where a greedy policy
can look excellent while learning nothing.

Compounds arrive as SMILES strings (text, e.g. `CC(=O)Oc1ccccc1C(=O)O` for
aspirin), which a model cannot consume. We convert them to **Morgan
fingerprints**: for every atom, describe the neighbourhood within 2 bonds, hash
each description into one of 512 buckets, set that bit. Molecules sharing
substructures share bits. This is the standard representation for ligand-based
screening (Rogers & Hahn, 2010).

In [ ]:
tox21, tox21_info = load_tox21("NR-AR", n_bits=512, radius=2)
print({k: v for k, v in tox21_info.items() if k != "feature_cols"})
tox21[["mol_id", "smiles", "label"]].head()

### A note on how model quality is measured

PR-AUC (area under the precision-recall curve) rather than ROC-AUC: with a 4%
hit rate, a model can have a fine ROC-AUC while its top 96 predictions are
nearly all inactive, because ROC-AUC averages over thresholds nobody will act
on.

And it is computed on a **fixed 20% holdout that no policy can select.** This
is not fussiness. If you score the model on whatever is left unselected, a
successful policy is punished for its own success: greedy harvests the easy
actives, so the leftovers are the hard ones. We measured it -- scored that way,
greedy gets PR-AUC 0.037 and random gets 0.485, while greedy finds eight times
as many actives. That number describes the difficulty of the leftovers, not the
quality of the model. A fixed holdout keeps the measuring stick still.

In [ ]:
tox21_frame, tox21_results = compare_policies(
    tox21,
    label_col="label",
    model_factory=forest,
    policies=POLICIES,
    seeds=SEEDS,
    feature_cols=tox21_info["feature_cols"],
    id_col="mol_id",
    budget=96,
    n_rounds=8,
    n_initial_labels=96,
    holdout_fraction=0.2,
    verbose=True,
)
TABDIR.mkdir(parents=True, exist_ok=True)
tox21_frame.to_csv(TABDIR / "02_tox21_nr_ar_rounds.csv", index=False)

In [ ]:
n_actives_tox21 = tox21_results["greedy"][0].n_actives
fig = plot_comparison(
    tox21_frame,
    n_actives=n_actives_tox21,
    title=("Tox21 NR-AR: 7,258 compounds, 4.2% hit rate, "
           "96 wells x 8 rounds, 5 seeds (band = min-max)"),
)
FIGDIR.mkdir(parents=True, exist_ok=True)
fig.savefig(FIGDIR / "02_headline_tox21_cumulative_hits.png", dpi=150, bbox_inches="tight")

In [ ]:
def final_table(frame):
    last = frame[frame["round"] == frame["round"].max()]
    return last.groupby("policy").agg(
        hits=("cumulative_hits", "mean"),
        hits_sd=("cumulative_hits", "std"),
        found=("fraction_of_achievable", "mean"),
        pr_auc=("pr_auc", "mean"),
        pr_auc_sd=("pr_auc", "std"),
    ).round(3).sort_values("hits", ascending=False)

final_table(tox21_frame)

### Read that table honestly

Greedy is at or near the top on hits, and the final PR-AUC column is flat
across every model-driven policy -- the differences are smaller than the
seed-to-seed standard deviation. **On this dataset, at this budget, the
information-seeking policies do not buy a better model, and they do not buy
more hits.**

The only policy clearly separated is `random`, which is worse at both. That is
worth stating plainly: the single largest effect here is "use the model at
all", and everything after that is a second-order choice.

So where does exploration earn anything? Look at the *marginal* productivity
rather than the cumulative total.

In [ ]:
per_round = tox21_frame.pivot_table(index="round", columns="policy", values="hit_rate")
fig, ax = plt.subplots(figsize=(7.5, 4.2))
for policy in per_round.columns:
    ax.plot(per_round.index, per_round[policy] * 100, "o-", label=policy, ms=4)
ax.set_xlabel("round"); ax.set_ylabel("hit rate this round (%)")
ax.set_title("Tox21 NR-AR: per-round productivity (mean of 5 seeds)")
ax.legend(fontsize=8, ncol=2)
fig.tight_layout()
fig.savefig(FIGDIR / "02_tox21_per_round_hit_rate.png", dpi=150, bbox_inches="tight")
(per_round * 100).round(1)

Greedy's hit rate rises, peaks, and falls. It is consuming a chemotype: the
model knows one region of chemical space, that region has a finite number of
members, and every well spent there confirms what the model already believed.
The diversity-constrained policy's rate is flat or rising over the same rounds.

With 8 rounds of 96 (768 of 7,258 compounds) greedy never runs out. Extend the
horizon and the picture changes -- that experiment is in the README.

## Dataset 2: BACE-1, a dose-response screen

1,513 inhibitors of human beta-secretase 1, each with a measured IC50
(Subramanian et al., 2016). BACE-1 is the enzyme that makes the first cut in
amyloid precursor protein, so it has been an Alzheimer's target for two
decades.

This dataset differs from Tox21 in three ways that matter:

1. **Potency is continuous.** `pIC50` is the negative log10 of the IC50
   concentration: 7.0 means 100 nM, and each unit is a tenfold improvement. We
   threshold at 7.0 to define a "hit", which gives a 44.6% hit rate -- and that
   number is a *choice*. Where you put the threshold decides your hit rate, so
   a campaign's apparent success is partly a decision about bookkeeping.
2. **It is pre-enriched.** These are published inhibitors of one target, not a
   diverse library, so the chemistry is narrow.
3. **It ships with 590 computed physicochemical descriptors**, so no chemistry
   toolkit is needed.

The budget here is 24 wells per round over 10 rounds -- a small-n campaign, the
case Loopwise is named for.

In [ ]:
bace, bace_info = load_bace(potency_threshold=7.0, use_descriptors=True)
print({k: v for k, v in bace_info.items() if k != "feature_cols"})
bace[["compound_id", "pIC50", "label"]].describe().round(2)

In [ ]:
bace_frame, bace_results = compare_policies(
    bace,
    label_col="label",
    model_factory=forest,
    policies=POLICIES,
    seeds=SEEDS,
    feature_cols=bace_info["feature_cols"],
    id_col="compound_id",
    budget=24,
    n_rounds=10,
    n_initial_labels=24,
    holdout_fraction=0.2,
    verbose=True,
)
bace_frame.to_csv(TABDIR / "02_bace_rounds.csv", index=False)
final_table(bace_frame)

In [ ]:
fig = plot_comparison(
    bace_frame,
    n_actives=bace_results["greedy"][0].n_actives,
    title=("BACE-1 dose-response: 1,513 compounds, 44.6% hit rate at pIC50>=7, "
           "24 wells x 10 rounds, 5 seeds"),
)
fig.savefig(FIGDIR / "02_bace_cumulative_hits.png", dpi=150, bbox_inches="tight")

### The trade-off, finally visible

On BACE the two objectives come apart cleanly. Greedy finds far more hits than
BALD *and* ends with the worst model of any policy -- worse than `random`.

That is the trade-off this package is about, measured rather than asserted. It
also makes the choice concrete: if the campaign's deliverable is a shortlist of
actives, greedy. If the deliverable is a model you will use on the next target,
greedy is destroying the thing you are paying for.

In [ ]:
last = bace_frame[bace_frame["round"] == bace_frame["round"].max()]
agg = last.groupby("policy").agg(hits=("cumulative_hits", "mean"),
                                 pr_auc=("pr_auc", "mean"))
fig, ax = plt.subplots(figsize=(6.2, 4.6))
for policy, row in agg.iterrows():
    ax.scatter(row["hits"], row["pr_auc"], s=70)
    ax.annotate(policy, (row["hits"], row["pr_auc"]),
                textcoords="offset points", xytext=(7, 3), fontsize=9)
ax.set_xlabel("cumulative hits found (mean of 5 seeds)")
ax.set_ylabel("final PR-AUC on the never-selected holdout")
ax.set_title("BACE-1: hits now vs model quality later\n(up and to the right is better; nothing is there)")
ax.grid(alpha=0.3)
fig.tight_layout()
fig.savefig(FIGDIR / "02_bace_hits_vs_model_quality.png", dpi=150, bbox_inches="tight")
agg.round(3).sort_values("hits", ascending=False)

## Rounds to reach half the achievable hits

The operational question is rarely "how many actives in total" -- it is "how
many plates until I have enough to move on". `rounds_to_fraction` answers it.

In [ ]:
rows = []
for name, dataset in (("Tox21 NR-AR", tox21_results), ("BACE-1", bace_results)):
    for policy, runs in dataset.items():
        reached = [r.rounds_to_fraction(0.5) for r in runs]
        got = [r for r in reached if r is not None]
        rows.append({
            "dataset": name,
            "policy": policy,
            "rounds_to_50pct": np.mean(got).round(1) if got else None,
            "seeds_reaching_50pct": f"{len(got)}/{len(runs)}",
            "final_pr_auc": round(float(np.mean([r.final_pr_auc for r in runs])), 3),
        })
pd.DataFrame(rows).pivot(index="policy", columns="dataset",
                         values=["rounds_to_50pct", "final_pr_auc"])

## Does a noisy assay change the answer?

Everything above assumes the measurement is correct. Real assays are not. With
`label_noise=0.2`, each revealed label has a 20% chance of being flipped when
the model sees it -- while scoring still uses the true labels, so we measure how
much of a policy's advantage survives.

In [ ]:
noisy_frame, noisy_results = compare_policies(
    tox21,
    label_col="label",
    model_factory=forest,
    policies=["random", "greedy", "bald", "hybrid"],
    seeds=(0, 1, 2),
    feature_cols=tox21_info["feature_cols"],
    id_col="mol_id",
    budget=96,
    n_rounds=6,
    n_initial_labels=96,
    label_noise=0.2,
    verbose=False,
)
clean_frame, _ = compare_policies(
    tox21,
    label_col="label",
    model_factory=forest,
    policies=["random", "greedy", "bald", "hybrid"],
    seeds=(0, 1, 2),
    feature_cols=tox21_info["feature_cols"],
    id_col="mol_id",
    budget=96,
    n_rounds=6,
    n_initial_labels=96,
    verbose=False,
)
comparison = pd.DataFrame({
    "clean": final_table(clean_frame)["hits"],
    "20% label noise": final_table(noisy_frame)["hits"],
})
comparison["lost to noise"] = (comparison["clean"] - comparison["20% label noise"]).round(1)
comparison.round(1)

## What replay cannot tell you

Worth being explicit, because this notebook is the repo's main claim:

1. **The pool is a measured library, not yours.** Tox21 compounds were selected
   for a screening programme; a vendor catalogue is less information-rich and
   its hit rate is lower.
2. **No synthesis delay, no attrition, no compound availability.** Round 4 here
   happens instantly; in a lab it happens in six weeks.
3. **The featurization is frozen.** Real campaigns change their features as
   they learn the chemistry, and that often matters more than the selection
   policy does.
4. **One dataset is one dataset.** We ran two, they disagreed, and the honest
   conclusion is the conditional one: *which* policy wins depends on the hit
   rate, the budget-to-pool ratio, the horizon, and which deliverable you are
   paying for.

Run `loopwise replay tox21 --n-seeds 5` to reproduce the Tox21 panel from the
command line.